In [1]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()
PROJECT_ROOT

WindowsPath('C:/AirPollutionPrediction-CNN-BiLSTM')

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import tensorflow as tf
load_model = tf.keras.models.load_model

from models.cnn_bilstm import train_cnn_bilstm, predict_cnn_bilstm

In [3]:
PROJECT_ROOT = Path.cwd().parent   
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

STATION_SPLIT_DIR = DATA_DIR / "station_split_24havg"   
pm25_scaler = joblib.load(ARTIFACTS_DIR / "pm25_scaler.pkl")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STATION_SPLIT_DIR exists:", STATION_SPLIT_DIR.exists())
print("Loaded pm25_scaler:", ARTIFACTS_DIR / "pm25_scaler.pkl")


PROJECT_ROOT: c:\AirPollutionPrediction-CNN-BiLSTM
STATION_SPLIT_DIR exists: True
Loaded pm25_scaler: c:\AirPollutionPrediction-CNN-BiLSTM\artifacts\pm25_scaler.pkl


c:\Users\KimNgan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
target_col = "PM2.5_24h_avg"
id_cols = ["Station_No", "date"]

SEQ_LEN = 48
EPOCHS = 200
BATCH = 64
PATIENCE = 20
LR = 3e-4
LSTM_UNITS = 128

print("Target:", target_col)
print("SEQ_LEN:", SEQ_LEN)


Target: PM2.5_24h_avg
SEQ_LEN: 48


In [5]:
import numpy as np
from sklearn.metrics import r2_score

def metrics_real_from_scaled(y_true_scaled, y_pred_scaled, scaler, eps=1e-6, mape_threshold=1.0):
    # Đảm bảo đầu vào là mảng phẳng
    y_true_scaled = np.asarray(y_true_scaled).reshape(-1)
    y_pred_scaled = np.asarray(y_pred_scaled).reshape(-1)

    # Giải mã (Inverse Transform) về đơn vị thực tế (µg/m³)
    yt = scaler.inverse_transform(y_true_scaled.reshape(-1,1)).ravel()
    yp = scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).ravel()

    # Tính toán các chỉ số cơ bản
    diff = yp - yt
    mae = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))

    # --- TÍNH R^2 SCORE ---
    # R2 = 1 là dự báo hoàn hảo. R2 = 0 là dự báo bằng giá trị trung bình.
    r2 = float(r2_score(yt, yp))

    # Tính MAPE với ngưỡng loại bỏ giá trị nhỏ (tránh chia cho ~0)
    mask = np.abs(yt) >= mape_threshold
    mape = float(np.mean(np.abs(diff[mask]) / np.abs(yt[mask]))) if np.any(mask) else float("nan")

    # Tính Hệ số tương quan Pearson (Correlation)
    if np.std(yt) < eps or np.std(yp) < eps:
        corr = float("nan")
    else:
        corr = float(np.corrcoef(yt, yp)[0,1])

    return {
        "Correlation": corr, 
        "R2": r2,           # Chỉ số mới
        "RMSE": rmse, 
        "MAPE": mape, 
        "MAE": mae
    }

In [6]:
station_dirs = sorted([p for p in STATION_SPLIT_DIR.glob("station_*") if p.is_dir()],
                      key=lambda p: int(p.name.split("_")[1]))

print("Found stations:", [p.name for p in station_dirs])


Found stations: ['station_1', 'station_2', 'station_3', 'station_4', 'station_5', 'station_6']


In [7]:
st_dir = STATION_SPLIT_DIR / "station_1"

train_df = pd.read_csv(st_dir / "train.csv")
val_df   = pd.read_csv(st_dir / "val.csv")
test_df  = pd.read_csv(st_dir / "test.csv")

feature_cols = [c for c in train_df.columns if c not in (id_cols + [target_col])]

X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df[target_col].values.astype(np.float32)

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df[target_col].values.astype(np.float32)

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df[target_col].values.astype(np.float32)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Num features:", len(feature_cols))


Shapes: (8096, 33) (1734, 33) (1736, 33)
Num features: 33


In [ ]:
ckpt = ARTIFACTS_DIR / f"cnn_bilstm_station1_24havg_seq{SEQ_LEN}_u{LSTM_UNITS}.keras"

model, history = train_cnn_bilstm(
    X_train, y_train,
    X_val, y_val,
    seq_len=SEQ_LEN,
    epochs=EPOCHS,
    batch_size=BATCH,
    use_early_stopping=True,
    patience=PATIENCE,
    learning_rate=LR,
    lstm_units=LSTM_UNITS,
    use_attention=True,
    checkpoint_path=str(ckpt)
)

best = load_model(ckpt)

y_pred_scaled = predict_cnn_bilstm(best, X_test, seq_len=SEQ_LEN)
y_true_scaled = y_test[SEQ_LEN:]

m1 = metrics_real_from_scaled(y_true_scaled, y_pred_scaled, pm25_scaler, eps=1.0)

print("\nStation 1 - 24hAVG (REAL SCALE µg/m³):")
for k,v in m1.items():
    print(f"{k}: {v}")


Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 0.0957
Epoch 1: val_loss improved from None to 0.00265, saving model to c:\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras

Epoch 1: finished saving model to c:\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 21s 112ms/step - loss: 0.0353 - val_loss: 0.0027 - learning_rate: 3.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - loss: 0.0052
Epoch 2: val_loss improved from 0.00265 to 0.00213, saving model to c:\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras

Epoch 2: finished saving model to c:\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 15s 120ms/step - loss: 0.0048 - val_loss: 0.0021 - learning_rate: 3.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - loss: 0.0041
Epoch 3: val

In [10]:
import keras
import pandas as pd
from sklearn.metrics import r2_score

# Danh sách để lưu kết quả của từng trạm
summary_list = []

for i in range(1, 7): # Chạy từ 1 đến 6
    ckpt = rf'C:\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station{i}_24havg_seq48_u128.keras'
    print(f"Đang xử lý: Station {i}...", end="\r")
    
    try:
        # 1. Load model
        model = keras.models.load_model(ckpt)

        # 2. Dự đoán
        y_pred_scaled = predict_cnn_bilstm(model, X_test, seq_len=SEQ_LEN)
        y_true_scaled = y_test[SEQ_LEN:]

        # 3. Tính toán Metrics (hàm của bạn)
        m = metrics_real_from_scaled(y_true_scaled, y_pred_scaled, pm25_scaler, eps=1.0)
        
        # 4. Thêm tên trạm vào kết quả
        m['Station'] = f"Station {i}"
        
        # Thêm R2_Score nếu chưa có
        if 'R2_Score' not in m:
            m['R2_Score'] = r2_score(y_true_scaled, y_pred_scaled)

        # 5. Lưu vào danh sách tổng
        summary_list.append(m)
        
        # Xóa model để nhẹ máy
        keras.backend.clear_session()
        
    except Exception as e:
        print(f"\nLỗi tại Station {i}: {e}")

# --- XUẤT KẾT QUẢ ---
# Chuyển danh sách thành DataFrame
df_results = pd.DataFrame(summary_list)

# Đưa cột 'Station' lên đầu bảng
cols = ['Station'] + [c for c in df_results.columns if c != 'Station']
df_results = df_results[cols]

print("\n" + "="*50)
print("BẢNG TỔNG HỢP KẾT QUẢ 6 TRẠM")
print("="*50)
print(df_results.to_string(index=False)) # to_string giúp hiển thị toàn bộ không bị cắt

# Tùy chọn: Lưu ra file Excel để nộp báo cáo
# df_results.to_excel("Ket_qua_6_station.xlsx", index=False)

Đang xử lý: Station 6...
BẢNG TỔNG HỢP KẾT QUẢ 6 TRẠM
  Station  Correlation       R2      RMSE     MAPE       MAE  R2_Score
Station 1     0.985773 0.968292 10.531547 0.102112  7.101685  0.968292
Station 2     0.985282 0.956566 12.326155 0.129841  8.234040  0.956566
Station 3     0.980081 0.923389 16.370243 0.152265 14.574527  0.923389
Station 4     0.881140 0.512455 41.296982 0.380498 38.055485  0.512455
Station 5     0.976746 0.938069 14.718525 0.130358 10.868428  0.938069
Station 6     0.972603 0.907946 17.944580 0.196480 14.325096  0.907946


In [12]:
all_rows = []

for st_dir in station_dirs:
    st = int(st_dir.name.split("_")[1])

    train_df = pd.read_csv(st_dir / "train.csv")
    val_df   = pd.read_csv(st_dir / "val.csv")
    test_df  = pd.read_csv(st_dir / "test.csv")

    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32)

    X_val = val_df[feature_cols].values.astype(np.float32)
    y_val = val_df[target_col].values.astype(np.float32)

    X_test = test_df[feature_cols].values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32)

    ckpt = ARTIFACTS_DIR / f"cnn_bilstm_station{st}_24havg_seq{SEQ_LEN}_u{LSTM_UNITS}.keras"

    model, _ = train_cnn_bilstm(
        X_train, y_train,
        X_val, y_val,
        seq_len=SEQ_LEN,
        epochs=EPOCHS,
        batch_size=BATCH,
        use_early_stopping=True,
        patience=PATIENCE,
        learning_rate=LR,
        lstm_units=LSTM_UNITS,
        use_attention=True,
        checkpoint_path=str(ckpt)
    )

    best = load_model(ckpt)

    y_pred_scaled = predict_cnn_bilstm(best, X_test, seq_len=SEQ_LEN)
    y_true_scaled = y_test[SEQ_LEN:]

    m = metrics_real_from_scaled(y_true_scaled, y_pred_scaled, pm25_scaler, eps=1.0)
    row = {"Station_No": st, "n_test": int(len(y_true_scaled))}
    row.update(m)
    all_rows.append(row)

    print(f"Station {st} | Corr={m['Correlation']:.3f} RMSE={m['RMSE']:.2f} MAE={m['MAE']:.2f} MAPE={m['MAPE']:.3f}")

results_df = pd.DataFrame(all_rows).sort_values("Station_No").reset_index(drop=True)
results_df


Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - loss: 0.1095
Epoch 1: val_loss improved from None to 0.00317, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 67s 357ms/step - loss: 0.0381 - val_loss: 0.0032 - learning_rate: 3.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 513ms/step - loss: 0.0051
Epoch 2: val_loss improved from 0.00317 to 0.00304, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 109s 573ms/step - loss: 0.0048 - val_loss: 0.0030 - learning_rate: 3.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0041
Epoch 3: val_loss improved from 0.00304 to 0.00204, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_24havg_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 162s 1s/step - loss: 0.0038 - val_loss: 0.00

,Station_No,n_test,Correlation,RMSE,MAPE,MAE
0,1,1688,0.985773,10.531547,0.102112,7.101685
1,2,1688,0.995178,3.207941,0.019301,2.395723
2,3,1688,0.996437,4.629591,0.031493,3.398873
3,4,1688,0.995880,5.245996,0.022811,3.982021
4,5,1688,0.982340,12.117559,0.120535,7.660173
5,6,1685,0.995141,5.275030,0.031820,4.194633


In [13]:
print("\nMean across stations:")
print(results_df[["Correlation","RMSE","MAE","MAPE"]].mean())

print("\nMAPE median:", results_df["MAPE"].median())


Mean across stations:
Correlation    0.991791
RMSE           6.834611
MAE            4.788851
MAPE           0.054679
dtype: float64

MAPE median: 0.031656717881560326


In [14]:
paper_table2 = pd.DataFrame({
    "Station_No":[1,2,3,4,5,6],
    "RMSE":[9.58,9.47,16.63,15.67,7.30,10.03],
    "MAPE":[0.54,0.34,0.54,0.39,0.31,0.31],
    "MAE":[7.12,6.35,11.09,9.44,4.89,6.00],
    "Correlation":[0.34,0.49,0.50,0.37,0.46,0.44]
})

paper_table1 = pd.DataFrame({
    "Station_No":[1,2,3,4,5,6],
    "RMSE":[7.22,7.72,12.34,9.83,4.85,5.94],
    "MAPE":[0.27,0.23,0.33,0.24,0.20,0.20],
    "MAE":[5.32,4.76,8.01,6.37,3.36,3.94],
    "Correlation":[0.70,0.71,0.69,0.71,0.72,0.77]
})

paper_table1, paper_table2


(   Station_No   RMSE  MAPE   MAE  Correlation
 0           1   7.22  0.27  5.32         0.70
 1           2   7.72  0.23  4.76         0.71
 2           3  12.34  0.33  8.01         0.69
 3           4   9.83  0.24  6.37         0.71
 4           5   4.85  0.20  3.36         0.72
 5           6   5.94  0.20  3.94         0.77,
    Station_No   RMSE  MAPE    MAE  Correlation
 0           1   9.58  0.54   7.12         0.34
 1           2   9.47  0.34   6.35         0.49
 2           3  16.63  0.54  11.09         0.50
 3           4  15.67  0.39   9.44         0.37
 4           5   7.30  0.31   4.89         0.46
 5           6  10.03  0.31   6.00         0.44)

rolling mean

In [15]:
cmp1 = results_df.merge(paper_table1, on="Station_No", suffixes=("_yours", "_paper1"))
cmp1["RMSE_gain"] = cmp1["RMSE_paper1"] - cmp1["RMSE_yours"]
cmp1["MAE_gain"]  = cmp1["MAE_paper1"]  - cmp1["MAE_yours"]
cmp1["Corr_gain"] = cmp1["Correlation_yours"] - cmp1["Correlation_paper1"]
cmp1[["Station_No","RMSE_yours","RMSE_paper1","RMSE_gain","MAE_yours","MAE_paper1","MAE_gain","Correlation_yours","Correlation_paper1","Corr_gain","MAPE_yours","MAPE_paper1"]]


,Station_No,RMSE_yours,RMSE_paper1,RMSE_gain,MAE_yours,MAE_paper1,MAE_gain,Correlation_yours,Correlation_paper1,Corr_gain,MAPE_yours,MAPE_paper1
0,1,10.531547,7.22,-3.311547,7.101685,5.32,-1.781685,0.985773,0.70,0.285773,0.102112,0.27
1,2,3.207941,7.72,4.512059,2.395723,4.76,2.364277,0.995178,0.71,0.285178,0.019301,0.23
2,3,4.629591,12.34,7.710409,3.398873,8.01,4.611127,0.996437,0.69,0.306437,0.031493,0.33
3,4,5.245996,9.83,4.584004,3.982021,6.37,2.387979,0.995880,0.71,0.285880,0.022811,0.24
4,5,12.117559,4.85,-7.267559,7.660173,3.36,-4.300173,0.982340,0.72,0.262340,0.120535,0.20
5,6,5.275030,5.94,0.664970,4.194633,3.94,-0.254633,0.995141,0.77,0.225141,0.031820,0.20


Mean compare

In [16]:
mean_yours = results_df[["Correlation","RMSE","MAE","MAPE"]].mean()
mean_p1 = paper_table1[["Correlation","RMSE","MAE","MAPE"]].mean()
mean_p2 = paper_table2[["Correlation","RMSE","MAE","MAPE"]].mean()

summary_compare = pd.DataFrame({
    "Yours_24havg": mean_yours,
    "Paper_Table1_with_rollmean": mean_p1,
    "Paper_Table2_no_rollmean": mean_p2
})
summary_compare


,Yours_24havg,Paper_Table1_with_rollmean,Paper_Table2_no_rollmean
Correlation,0.991791,0.716667,0.433333
RMSE,6.834611,7.983333,11.446667
MAE,4.788851,5.293333,7.481667
MAPE,0.054679,0.245000,0.405000


Kiểm tra có bị overlift kh 

In [28]:
from pathlib import Path
import pandas as pd

STATION_SPLIT_DIR = STATION_SPLIT_DIR  # đã define trước

for st_dir in sorted(STATION_SPLIT_DIR.glob("station_*"),
                     key=lambda p: int(p.name.split("_")[1])):

    st = int(st_dir.name.split("_")[1])

    train_df = pd.read_csv(st_dir / "train.csv")
    val_df   = pd.read_csv(st_dir / "val.csv")
    test_df  = pd.read_csv(st_dir / "test.csv")

    train_df["date"] = pd.to_datetime(train_df["date"])
    val_df["date"]   = pd.to_datetime(val_df["date"])
    test_df["date"]  = pd.to_datetime(test_df["date"])

    print(f"\nStation {st}")
    print("Train:", train_df["date"].min(), "->", train_df["date"].max())
    print("Val  :", val_df["date"].min(),   "->", val_df["date"].max())
    print("Test :", test_df["date"].min(),  "->", test_df["date"].max())

    # kiểm tra overlap
    assert train_df["date"].max() < val_df["date"].min(), "Train/Val overlap!"
    assert val_df["date"].max()   < test_df["date"].min(), "Val/Test overlap!"

print("\n Time-based split OK (no shuffle, no overlap)")



Station 1
Train: 2021-02-24 20:00:00 -> 2022-01-28 03:00:00
Val  : 2022-01-28 04:00:00 -> 2022-04-10 09:00:00
Test : 2022-04-10 10:00:00 -> 2022-06-21 17:00:00

Station 2
Train: 2021-02-24 20:00:00 -> 2022-01-28 03:00:00
Val  : 2022-01-28 04:00:00 -> 2022-04-10 09:00:00
Test : 2022-04-10 10:00:00 -> 2022-06-21 17:00:00

Station 3
Train: 2021-02-24 20:00:00 -> 2022-01-28 03:00:00
Val  : 2022-01-28 04:00:00 -> 2022-04-10 09:00:00
Test : 2022-04-10 10:00:00 -> 2022-06-21 17:00:00

Station 4
Train: 2021-02-24 20:00:00 -> 2022-01-28 03:00:00
Val  : 2022-01-28 04:00:00 -> 2022-04-10 09:00:00
Test : 2022-04-10 10:00:00 -> 2022-06-21 17:00:00

Station 5
Train: 2021-02-24 20:00:00 -> 2022-01-28 03:00:00
Val  : 2022-01-28 04:00:00 -> 2022-04-10 09:00:00
Test : 2022-04-10 10:00:00 -> 2022-06-21 17:00:00

Station 6
Train: 2021-02-25 15:00:00 -> 2022-01-28 08:00:00
Val  : 2022-01-28 09:00:00 -> 2022-04-10 12:00:00
Test : 2022-04-10 13:00:00 -> 2022-06-21 17:00:00

 Time-based split OK (no shuffle,

In [27]:
SEQ_LEN = 48  # đang dùng

for st_dir in sorted(STATION_SPLIT_DIR.glob("station_*"),
                     key=lambda p: int(p.name.split("_")[1])):

    st = int(st_dir.name.split("_")[1])

    train_df = pd.read_csv(st_dir / "train.csv")
    test_df  = pd.read_csv(st_dir / "test.csv")

    train_df["date"] = pd.to_datetime(train_df["date"])
    test_df["date"]  = pd.to_datetime(test_df["date"])

    last_train = train_df["date"].max()
    first_test = test_df["date"].min()

    gap_hours = (first_test - last_train).total_seconds() / 3600

    print(f"Station {st}: gap_hours =", gap_hours)

    if gap_hours < 0:
        print(" Overlap detected")
    else:
        print("OK")

print("\nNote: gap=1h là bình thường vì test dùng quá khứ từ test set, không từ train.")


Station 1: gap_hours = 1735.0
OK
Station 2: gap_hours = 1735.0
OK
Station 3: gap_hours = 1735.0
OK
Station 4: gap_hours = 1735.0
OK
Station 5: gap_hours = 1735.0
OK
Station 6: gap_hours = 1733.0
OK

Note: gap=1h là bình thường vì test dùng quá khứ từ test set, không từ train.


In [25]:
import pandas as pd

st_dir = STATION_SPLIT_DIR / "station_1"

train_df = pd.read_csv(st_dir / "train.csv")
train_df["date"] = pd.to_datetime(train_df["date"])
train_df = train_df.sort_values("date").reset_index(drop=True)

# tự tính rolling mean 24h
manual_roll = train_df["PM2.5"].rolling(window=24, min_periods=24).mean()

diff = (manual_roll - train_df["PM2.5_24h_avg"]).abs()

print("Max difference:", diff.max())


Max difference: 1.1102230246251565e-16


In [26]:
# kiểm tra feature leakage sơ bộ
st_dir = STATION_SPLIT_DIR / "station_1"

train_df = pd.read_csv(st_dir / "train.csv")
test_df  = pd.read_csv(st_dir / "test.csv")

target_col = "PM2.5_24h_avg"
id_cols = ["Station_No","date"]

feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]

corrs = []

for col in feature_cols:
    c = test_df[[col, target_col]].corr().iloc[0,1]
    corrs.append((col, c))

corrs_sorted = sorted(corrs, key=lambda x: abs(x[1]), reverse=True)

print("Top 10 feature correlations with target in test:")
for col, c in corrs_sorted[:10]:
    print(col, ":", c)


Top 10 feature correlations with target in test:
O3 : nan
NO2 : nan
SO2 : nan
PM2.5_roll12_mean : 0.8685471235295746
PM2.5_lag12 : 0.7994135797929961
PM2.5_lag6 : 0.7540451673194263
PM2.5_roll6_mean : 0.7305766032297427
PM2.5_lag3 : 0.6899184872163892
PM2.5_lag2 : 0.6607771252983659
PM2.5_roll3 : 0.6486054610619079
